In [1]:
import sys
sys.path.append('../src')
from data_validation import validate_dataframe
import glob
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError
import seaborn as sns 
from typing import Literal

IndexError: list index out of range

In [ ]:
first_20_path = r'C:\Users\Aamir Bin Abbas\Desktop\Research & Thesis\Stock-Price-Predictor-Close--LSTM\data\processed\yahoo_finance\first_20_companies'

In [ ]:
def get_dataset(filepath: str,
                file_num: int
                ) -> pd.DataFrame:
    """Helper function to accessing csv files in directory

    Args:
        filepath (str): directory path to containing csv files
        file_num (int): list index number of the file list

    Returns:
        pd.DataFrame: pandas dataframe
    """
    dataset_dir = glob.glob(f"{filepath}/*.csv")
    
    df = pd.read_csv(dataset_dir[file_num])
    
    return df

In [ ]:
def check_monotonic(
    dataset: pd.DataFrame) -> bool:
    """Check if dates are monotonic or not

    Args:
        dataset (pd.DataFrame): current pandas dataset(note: date column must be datetime format)
    
    Raises:
        EmptyDataError: raised when dataset is empty

    Returns:
        bool: True if monotonic else False
    """
        
    validate_dataframe(dataset)
    
    date_col = [col for col in dataset.columns]
    
    for column in date_col:
        
        if pd.api.types.is_datetime64_any_dtype(dataset[column]):
                                    
            is_increasing = dataset[column].is_monotonic_increasing
            
            is_decreasing = dataset[column].is_monotonic_decreasing
    
            return is_increasing or is_decreasing
    
    return False

In [ ]:
def convert_date(dataset: pd.DataFrame,
                date_col: str,
                errors: Literal['raise','coerce'] = 'coerce'
                ) -> pd.DataFrame:
    """Convert 'Date' column in the dataset.
        Warning :  Using errors='coerce' converts invalid date values to NaT.

    Args:
        dataset (pd.DataFrame): The input pandas DataFrame.
        date_col (str): Name of the column to convert. Case-sensitive.
        errors (Literal['raise', 'coerce']): Determines how invalid date values
        are handled. Defaults to 'coerce'.
        - 'raise': Raises an exception when an invalid date is encountered.
        - 'coerce': Converts invalid date values to NaT.
        
    Raises:
        EmptyDataError: raises error if dataset is empty
        KeyError: raises error if the date column is not present in the dataset

    Returns:
        pd.DataFrame: The DataFrame with the specified column converted to pandas datetime.
    """
    
    
    validate_dataframe(dataset)
    
    if date_col not in dataset.columns:
        raise KeyError(f'No {date_col} was found in the dataset')
    
    dataset[date_col] = pd.to_datetime(dataset[date_col],errors=errors) 
    
    return dataset

In [ ]:
def round_float(
    dataset: pd.DataFrame,
    r: int = 2
    ) -> pd.DataFrame:
    """
        Round floating-point columns in a Pandas DataFrame.

        Rounds all floating-point columns to the specified decimal
        precision. The default precision is 2 decimal places.

        Args:
            dataset (pd.DataFrame): The DataFrame to be modified.
            r (int): Number of decimal places to retain. Defaults to 2.

        Raises:
            TypeError: If dataset is not a DataFrame or r is not an integer.
            ValueError: If r is negative.
            EmptyDataError: If the provided DataFrame is empty.

        Returns:
            pd.DataFrame: The modified DataFrame with rounded float columns.
    """
    
    validate_dataframe(dataset)
    
    if isinstance(r,bool) or isinstance(r,float):
        raise TypeError(f'The arg {r} provided must be an integer, got {type(r).__name__}')
    
    if r < 0:
        raise ValueError("The rounding parameter 'r' must be 0 or greater.")

    columns = [col for col in dataset.columns]
    
    for col in columns:
        if pd.api.types.is_float_dtype(dataset[col]):
            dataset[col] = round(dataset[col],r)
    
    return dataset

In [ ]:
df = get_dataset(
    first_20_path,
    0)

pipeline = (
    round_float(df,2),
    convert_date(df,'Date')
)

In [ ]:
return_df = df.groupby(
    ['Date','Open','Close'],
    dropna=True).size().reset_index(name='Return')

In [ ]:
return_df['Return'] = ((df['Close'] - df['Open']) / df['Open']) * 100

In [ ]:
round_float(return_df )

In [ ]:
plt.scatter(
    x=return_df ['Open'],
    y=return_df ['Close'],
    c=return_df ['Close'],
    edgecolors='black',
    alpha=0.8)

plt.plot(
    [return_df ['Open'].min(), return_df ['Open'].max()], # min and max coordinates
    [return_df ['Open'].min(), return_df ['Open'].max()],
    color='red',
    linestyle='-', # type of line
    linewidth=2, # width of line
)
plt.title('Open vs Close')

plt.xlabel('Open')

plt.ylabel('Close')

plt.grid() 
#plt.savefig(r'C:\Users\Aamir Bin Abbas\Desktop\Research & Thesis\Stock-Price-Predictor-Close--LSTM\outputs\visualization\Linear_relation(Open_vs_Close)')

plt.show()

In [ ]:
def daily_pct_returns(
    dataset: pd.DataFrame,
    column: str = 'Close'
    ) -> pd.DataFrame:
    """Calculates the daily return percentage using 'Close'

    Args:
        dataset (pd.DataFrame): The original dataset 
        column (str, optional): The 'Close' column required for calculation. Defaults to 'Close'.

    Raises:
        TypeError: Raised if column or dataset type aer incorrect 
        EmptyDataError: Raised if dataset is empty
        KeyError: Raised if no such column was found
        
    Returns:
        pd.DataFrame: A pandas dataset containing original columns with daily pct return as appended on it 
    """
    validate_dataframe(dataset)

    if not isinstance(column,str):
        raise TypeError(f'The datatype of {column} must be str, got {type(column).__name__} ')
        
    clean_cols = [c.strip().lower() for c in dataset.columns]
    check_col = column.strip().lower()
    
    if check_col.strip().lower() not in clean_cols:
        raise KeyError(f'No {check_col} was found in the dataset')
    
    actual_column = dataset.columns[clean_cols.index(check_col)]
    
    
    # calculating the daily pct return
    daily_pct = [np.nan] # can be changed from 0 to np.nan , as the first value of 
    
    for i in range(1,dataset[actual_column].size):
        daily_pct.append(
            (
                (dataset[actual_column].iloc[i] - dataset[actual_column].iloc[i-1]) / dataset[actual_column].iloc[i-1]) * 100
            )
    
    dataset['daily_pct_return'] = daily_pct
    
    return dataset